# Phase 1 — Gap Analysis

**Goal:** Prove a meaningful, directionally predictive price gap exists between Vegas consensus probabilities and Kalshi implied probabilities on NBA, NFL, and MLB game contracts.

**Success criteria:**
1. Gap > 3% on at least 20% of sampled contracts
2. Vegas side wins > 50% of flagged games
3. Pattern is consistent across all three sports
4. Results are statistically significant (p < 0.05)

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from core.utils import kalshi_to_prob, remove_vig
from analysis.merger import merge_feeds
from analysis.gap_calculator import compute_gaps, flag_tradeable, GAP_THRESHOLD
from analysis.backtest import run_backtest
from dashboard.charts import plot_gap_distribution, plot_win_rate_by_threshold, plot_sport_breakdown

sns.set_theme(style='darkgrid')
os.makedirs('../outputs', exist_ok=True)
print('Imports OK')

## 1. Load Kalshi Data

In [ ]:
def load_kalshi(sport: str) -> pd.DataFrame:
    path = f'../data/raw/kalshi/{sport}_markets.json'
    with open(path) as f:
        markets = json.load(f)
    rows = []
    for m in markets:
        ticker = m.get('ticker', '')
        # Extract game date and teams from ticker or title
        rows.append({
            'ticker': ticker,
            'title': m.get('title', ''),
            'yes_price': m.get('last_price') or m.get('yes_ask'),
            'close_time': m.get('close_time'),
            'result': m.get('result'),
            'volume': m.get('volume'),
        })
    df = pd.DataFrame(rows)
    df['kalshi_home_prob'] = df['yes_price'].apply(lambda x: kalshi_to_prob(x) if x else None)
    df = df.dropna(subset=['kalshi_home_prob'])
    print(f'{sport.upper()}: {len(df)} Kalshi markets loaded')
    return df

kalshi_nba = load_kalshi('nba')
kalshi_nfl = load_kalshi('nfl')
kalshi_mlb = load_kalshi('mlb')
kalshi_nba.head(3)

## 2. Load Vegas Data

In [ ]:
def load_vegas(sport: str) -> pd.DataFrame:
    """Load all JSON snapshots for a sport and extract vig-free probabilities."""
    sport_dir = f'../data/raw/vegas/{sport}'
    rows = []
    for fname in os.listdir(sport_dir):
        if not fname.endswith('.json'):
            continue
        with open(os.path.join(sport_dir, fname)) as f:
            games = json.load(f)
        for game in games:
            home = game.get('home_team', '')
            away = game.get('away_team', '')
            game_date = game.get('commence_time', '')[:10]
            bookmakers = game.get('bookmakers', [])
            if not bookmakers:
                continue
            bk = next((b for b in bookmakers if b['key'] == 'pinnacle'), bookmakers[0])
            markets = {m['key']: m for m in bk.get('markets', [])}
            h2h = markets.get('h2h', {})
            outcomes = {o['name']: o['price'] for o in h2h.get('outcomes', [])}
            if home not in outcomes or away not in outcomes:
                continue
            home_prob, away_prob = remove_vig(outcomes[home], outcomes[away])
            rows.append({
                'game_date': game_date,
                'home_team': home,
                'away_team': away,
                'vegas_home_prob': home_prob,
                'vegas_away_prob': away_prob,
            })
    df = pd.DataFrame(rows).drop_duplicates(subset=['game_date', 'home_team', 'away_team'])
    print(f'{sport.upper()}: {len(df)} Vegas games loaded')
    return df

vegas_nba = load_vegas('nba')
vegas_nfl = load_vegas('nfl')
vegas_mlb = load_vegas('mlb')

## 3. Merge Feeds

In [ ]:
# TODO: parse game_date and teams from Kalshi ticker/title before merging
# merged_nba = merge_feeds(kalshi_nba, vegas_nba, 'nba')
# merged_nfl = merge_feeds(kalshi_nfl, vegas_nfl, 'nfl')
# merged_mlb = merge_feeds(kalshi_mlb, vegas_mlb, 'mlb')
print('Merge step — implement after inspecting Kalshi ticker format from pull_kalshi_markets.py')

## 4. Compute Gaps

In [ ]:
# combined = pd.concat([merged_nba, merged_nfl, merged_mlb], ignore_index=True)
# df_gaps = compute_gaps(combined)
# print(f'Total games: {len(df_gaps)}')
# print(f'Flagged (>{GAP_THRESHOLD:.0%}): {df_gaps["flagged"].sum()}')
# df_gaps.describe()

## 5. Gap Distribution Chart

In [ ]:
# plot_gap_distribution(df_gaps, output_path='../outputs/gap_distribution_chart.png')
# from IPython.display import Image
# Image('../outputs/gap_distribution_chart.png')

## 6. Backtest

In [ ]:
# results = run_backtest(df_gaps)
# print(f"Win rate: {results['win_rate']:.1%}")
# print(f"p-value:  {results['p_value']:.4f}")
# print(results['sport_breakdown'])